# KataGo Human SL 分析

重現 `aicup/analyze_humansl.py` 的分析，輸出每局在各 profile 下的落子機率，以及每手損失（`val648` 模式不算損失）。

- runtime：T4 GPU
- 外部資源：KataGo `v1.18.2`、`b18c384nbt-humanv0`、`kata1-b28c512nbt-s13255194368-d5935380940`，授權見報告的外部資源一節
- Drive 的 `MyDrive/aicup-2026/` 要有 `training-v1.1/` 的 10 個 CSV（官方訓練集 v1.1，含 `HD[n]`）與 `katago/` 的 2 個模型
- `val648` 模式另外要有 `humansl-analysis/val-players.csv`，欄位是 `player_id`、`rank`
- 規則、貼目、讓子依官方 Q&A，預設 `japanese`、6.5 目，`HD` 的處理見 `parse_game`
- Colab Secrets 要有唯讀的 `GH_TOKEN`

Drive 的做法：

- 不跑 `aicup.colab_setup`。原因：它會讀 `rank-features/`，曾讓 Drive 掛載斷線，這份分析用不到特徵。
- CSV 與模型先複製到 `/content`，分析期間不讀 Drive。
- 結果寫在 `/content`，每批結束後複製到 Drive。原因：直接寫 Drive 時，檔案開著的期間資料可能沒有同步，斷線後會遺失。
- 斷線時最多損失最近一批。重新開 runtime，依序執行所有 cell 就會接續。
- 分析中不要中斷 cell。

## 參數

`MODE` 選 `pilot`、`visits`、`main`、`val648`。

- `val648`：CNN val 切分中局數 20 以上的 648 位玩家，每人 20 局，3 個 profile，不算損失

In [ ]:
MODE = 'val648'
BATCH = 500

DRIVE = '/content/drive/MyDrive/aicup-2026'
RUNS = {  # (參數, 輸出檔名, 總局數)
    'pilot': ('--players-per-rank 1 --games-per-player 10', 'pilot-100-hd-v1.jsonl', 100),
    'visits': ('--players-per-rank 1 --games-per-player 10 --eval-visits 16', 'pilot-100-hd-v16.jsonl', 100),
    'main': ('--players-per-rank 30 --games-per-player 20', 'pilot-6000-hd.jsonl', 6000),
    'val648': ('--players-file /content/val-players.csv --games-per-player 20 '
               '--profiles rank_11k rank_1d rank_6d --no-eval', 'val648-p3.jsonl', 12960),
}
ARGS, OUT_NAME, TOTAL = RUNS[MODE]

## 取得程式

In [ ]:
import os, subprocess
from google.colab import drive, userdata

drive.mount('/content/drive')
REPO = '/content/aicup-2026-go'


def git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode:
        print(r.stderr.replace(userdata.get('GH_TOKEN'), '***'))
        raise SystemExit('git %s 失敗' % args[0])


if not os.path.exists(REPO):
    git('clone', '-q', 'https://%s@github.com/Jx-study/aicup-2026-go.git' % userdata.get('GH_TOKEN'), REPO)
os.chdir(REPO)
git('pull', '-q')
!pip install -q sgfmill
!git log --oneline -1
!nvidia-smi -L
os.environ['DRIVE'], os.environ['OUT_NAME'] = DRIVE, OUT_NAME

## 複製資料並安裝 KataGo

要看到 `10`、兩個模型與 `katago`，`val648` 模式還要看到 `val-players.csv`。Drive 上已有結果時會複製回本地，接續分析。

In [ ]:
%%bash
mkdir -p /content/training /content/models /content/katago
cp "$DRIVE"/training-v1.1/train_*.csv /content/training/
cp "$DRIVE"/katago/*.bin.gz /content/models/
if [ -f "$DRIVE/humansl-analysis/val-players.csv" ]; then
    cp "$DRIVE/humansl-analysis/val-players.csv" /content/
fi
if [ -f "$DRIVE/humansl-analysis/$OUT_NAME" ]; then
    cp "$DRIVE/humansl-analysis/$OUT_NAME" /content/
fi
cd /content/katago
ZIP=katago-v1.18.2-cuda12.8-cudnn9.8.0-linux-x64.zip
wget -q https://github.com/lightvector/KataGo/releases/download/v1.18.2/$ZIP
unzip -qjo $ZIP
chmod +x katago
ls /content/training | wc -l
ls -la /content/models /content/katago/katago analysis_example.cfg /content/*.csv

## 分析

每批結束會印出 Drive 上的行數，數字增加代表備份成功。

In [ ]:
LOCAL = '/content/' + OUT_NAME
LOG = '/content/%s.log' % MODE
BACKUP = DRIVE + '/humansl-analysis'
for n in sorted({*range(BATCH, TOTAL, BATCH), TOTAL}):
    !python -m aicup.analyze_humansl --csv '/content/training/train_*.csv' --model-dir /content/models {ARGS} --limit {n} --out {LOCAL} 2>&1 | tee -a {LOG}
    !mkdir -p {BACKUP} && cp {LOCAL} {LOG} {BACKUP}/ && wc -l {BACKUP}/{OUT_NAME}

## 釋放 runtime

跑完才執行。程式結束後 runtime 不會自動中斷，閒置也會扣額度。

In [ ]:
from google.colab import drive, runtime

drive.flush_and_unmount()
runtime.unassign()